In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd() + "/" + "src"))
from transformers import RagTokenizer, RagRetriever, RagSequenceForGeneration
import torch


In [ ]:
modelPath = "../models/rag-sequence-nq"

tokenizer = RagTokenizer.from_pretrained(modelPath)
# retriever = RagRetriever.from_pretrained(modelPath, index_name="exact", use_dummy_dataset=True)

input_dict = tokenizer.prepare_seq2seq_batch(
    "How many people live in Paris?", "In Paris, there are 10 million people.", return_tensors="pt"
)
input_ids = input_dict["input_ids"]

model: RagSequenceForGeneration = RagSequenceForGeneration.from_pretrained(modelPath)

In [ ]:
os.environ['HF_ENDPOINT']='https://hf-mirror.com'
retriever = RagRetriever.from_pretrained(modelPath, index_name="exact", use_dummy_dataset=True) 

In [11]:
# stateDict = model.state_dict()
# import os
# import torch
# import ctypes

# modelParamInfo = "model_info.txt"
# modelParam = "model.bin"

# with open(modelParamInfo, "w") as file, open(modelParam, "wb") as binFile:
#     for item in stateDict:
#         tensor: torch.Tensor = stateDict[item]
#         tensor = tensor.type(torch.float32)
#         file.write(item)
#         file.write(str(tensor.dtype))
#         file.write("\n")
#         file.write(str(stateDict[item].shape))
#         file.write("\n")
#         if not tensor.is_contiguous():
#             print("tensor is not contiguous")
#         file.write("\n")
#             tensor = tensor.contiguous()
#         data_ptr = tensor.data_ptr()
#         element_size = tensor.element_size()
#         total_size = tensor.numel() * element_size
#         byte_data = (ctypes.c_char * total_size).from_address(data_ptr)
#         binFile.write(byte_data)
# tensor = stateDict["rag.question_encoder.question_encoder.bert_model.embeddings.word_embeddings.weight"][0]
# print(f"{tensor[-3]:.6f}")

In [ ]:
import torch
from torch import nn, Tensor
import torch.nn.functional as F

torch.ops.load_library("/home/huahua/Projects/transformers-3.3.1/lib/libSecureRAGExtension.so")
SecureRAGExtension = torch.ops.SecureRAGExtension


SecureRAGExtension.openSGX()


class SecureLinear(nn.Linear):
    def forward(self, input: Tensor) -> Tensor:
        return SecureRAGExtension.secureLinear(input, self.weight, self.bias)

In [6]:
def secure(model: nn.Module):
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            # print(name)
            securelinear = SecureLinear(module.in_features, module.out_features)
            securelinear.weight.data = module.weight.clone()
            securelinear.bias.data = module.bias.clone()
            setattr(model, name, securelinear)
        if len(list(module.named_children())) > 0:
            secure(module)


secure(model)

In [ ]:
# 1. Encode
question_hidden_states = model.question_encoder(input_ids)[0]
# 2. Retrieve
docs_dict = retriever(input_ids.numpy(), question_hidden_states.detach().numpy(), return_tensors="pt")
doc_scores = torch.bmm(question_hidden_states.unsqueeze(1), docs_dict["retrieved_doc_embeds"].float().transpose(1, 2)).squeeze(1)
# 3. Forward to generator
outputs = model(context_input_ids=docs_dict["context_input_ids"], context_attention_mask=docs_dict["context_attention_mask"], doc_scores=doc_scores, decoder_input_ids=input_dict["labels"])